# Сcылки:</br>
## 1. https://pypi.org/project/bertopic/

In [48]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import nltk
from nltk.corpus import stopwords
import string
import re
import numpy as np
from tqdm import tqdm
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from sklearn.cluster import KMeans
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import MaximalMarginalRelevance
from bertopic.representation import TextGeneration
from transformers import pipeline

# 1. Загрузка датасета

In [49]:
df = pd.read_json("data/data.txt", lines=True)
df

,text,tags,schema_name,table_name
0,"1 1 1 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47923
1,"1 1 1 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710
2,"2 2 2 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710
3,"3 3 3 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710
4,"4 4 4 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710
...,...,...,...,...
18419,996 996 982 20231117_110117.jpg 2023-11-17 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876
18420,997 997 983 20231121_100442.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876
18421,998 998 984 20231121_095847.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876
18422,99 99 101 117 4.jpg 2023-11-16 00:00:00+00 13:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876


# 2. Очистка данных

In [50]:
# Загрузка стоп-слов (один раз при старте)
nltk.download('stopwords')
stop_words = set(stopwords.words('russian')) # Загрузка стоп-слов (один раз при старте)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [51]:
# Функция для очистки текста
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # Удаление URL
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text, flags=re.MULTILINE)
    # Удаление цифр и специальных символов
    text = re.sub(r'\d+', ' ', text)
    # Удаление пунктуации
    # text = text.translate(str.maketrans('', '', string.punctuation))
    translator = str.maketrans(string.punctuation, ' ' * len(string.punctuation))
    text = text.translate(translator)
    # Приведение к нижнему регистру (не всегда  обязательно, возможно ухудшит качество)
    text = text.lower()
    # Удаление стоп-слов (не всегда  обязательно, возможно ухудшит качество)
    stop_words = set(stopwords.words('russian'))
    words = text.split()
    words = [word for word in words if word not in stop_words and len(word) > 2]
    return ' '.join(words)

In [52]:
# Функция для получения имени сервиса. Именем сервиса является номер на 2-ой позиции в списке тегов
def get_service_name(list_of_tags):
    # если недостаточное кол-во тегов -> нет сервиса -> возвращаем пустую строку
    if len(list_of_tags) < 2:
        return ''
    # service_name = list_of_tags[1].split()[0] # получаем номер
    service_name = list_of_tags[1] # получаем цклую строку
    return service_name

In [53]:
# Функция для обработки чанка с очисткой текста
def process_chunk(chunk):
    """Генерирует эмбеддинги для чанка с предварительной очисткой"""
    # Очищаем тексты
    cleaned_texts = chunk['text'].apply(clean_text).tolist()
    # получаем имена сервисов
    service_names = chunk['tags'].apply(get_service_name).tolist()
    
    # Добавляем результаты в чанк
    chunk['cleaned_text'] = cleaned_texts
    chunk['service_name'] = service_names
    return chunk

In [54]:
# Размер чанка (подбирайте под ваши ресурсы)
CHUNK_SIZE = 1000  
# Инициализация прогресс-бара
pbar = tqdm(total=len(df), desc="Обработка датафрейма")
# Обработка по чанкам
results = []
for i in range(0, len(df), CHUNK_SIZE):
    chunk = df.iloc[i:i + CHUNK_SIZE].copy()
    processed_chunk = process_chunk(chunk)
    results.append(processed_chunk)
    pbar.update(len(chunk))
pbar.close()

# Сборка финального датафрейма
final_df = pd.concat(results, ignore_index=True)

Обработка датафрейма: 100%|██████████| 18424/18424 [00:04<00:00, 3733.73it/s]


In [55]:
final_df['full_table_name'] = final_df['schema_name'] + '.' + final_df['table_name']
final_df

,text,tags,schema_name,table_name,cleaned_text,service_name,full_table_name
0,"1 1 1 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47923,часть территории национального парка лосиный о...,440 Особо охраняемые природные территории,_46379._47923
1,"1 1 1 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,часть территории национального парка лосиный о...,440 Особо охраняемые природные территории,_46379._58710
2,"2 2 2 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,часть территории национального парка лосиный о...,440 Особо охраняемые природные территории,_46379._58710
3,"3 3 3 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,часть территории национального парка лосиный о...,440 Особо охраняемые природные территории,_46379._58710
4,"4 4 4 Часть территории национального парка ""Ло...","[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_58710,часть территории национального парка лосиный о...,440 Особо охраняемые природные территории,_46379._58710
...,...,...,...,...,...,...,...
18419,996 996 982 20231117_110117.jpg 2023-11-17 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,jpg photo зелао комплексный заказник зеленогра...,440 Особо охраняемые природные территории,_46379._47876
18420,997 997 983 20231121_100442.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,jpg photo юзао юао пип битцевский лес участок ...,440 Особо охраняемые природные территории,_46379._47876
18421,998 998 984 20231121_095847.jpg 2023-11-21 00:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,jpg photo юзао юао пип битцевский лес участок ...,440 Особо охраняемые природные территории,_46379._47876
18422,99 99 101 117 4.jpg 2023-11-16 00:00:00+00 13:...,"[МКА. Геоданные, 440 Особо охраняемые природны...",_46379,_47876,jpg photo зелао комплексный заказник зеленогра...,440 Особо охраняемые природные территории,_46379._47876


# 3. Создание модели на основе bag-of-words

In [56]:
# we add this to remove stopwords
# vectorizer_model = CountVectorizer(ngram_range=(1, 3))
# model = BERTopic(
#     vectorizer_model=vectorizer_model,
#     language='multilingual', calculate_probabilities=True,
#     verbose=True
# )
sentence_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine')
cluster_model = KMeans(n_clusters=50)
vectorizer_model = CountVectorizer(stop_words=None, ngram_range=(1, 3), min_df=10)
ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)
mmr = MaximalMarginalRelevance(diversity=0.3)
prompt = "I have a topic described by the following keywords: [KEYWORDS]. And described by the following documents [DOCUMENTS]. Based on the previous keywords and documents, what is this topic about?"
# Create your representation model
generator = pipeline('text2text-generation', model='google/flan-t5-base')
text_representation_model = TextGeneration(generator)
representation_model = {
   "Main": main_representation,
   "Aspect1":  aspect_model1,
   "Aspect2":  aspect_model2
}
representation_models = [mmr]
model = BERTopic(
    embedding_model=sentence_model, umap_model=umap_model, calculate_probabilities=True, hdbscan_model=cluster_model,
    ctfidf_model=ctfidf_model, representation_model=representation_models, verbose=True)
topics, probs = model.fit_transform(final_df['cleaned_text'])


2025-09-04 16:14:07,193 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/576 [00:00<?, ?it/s]

2025-09-04 16:14:18,139 - BERTopic - Embedding - Completed ✓
2025-09-04 16:14:18,140 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-09-04 16:14:21,581 - BERTopic - Dimensionality - Completed ✓
2025-09-04 16:14:21,582 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-09-04 16:14:21,647 - BERTopic - Cluster - Completed ✓
2025-09-04 16:14:21,652 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-09-04 16:14:23,569 - BERTopic - Representation - Completed ✓


In [57]:
freq = model.get_topic_info()
freq

,Topic,Count,Name,Representation,Representative_Docs
0,0,1777,0_памятники_памятник_план_правительства,"[памятники, памятник, план, правительства, мог...",[природно исторический парк москворецкий приро...
1,1,1762,1_договор_аренды_правообладателю_земельном,"[договор, аренды, правообладателю, земельном, ...",[публичное акционерное общество московская объ...
2,2,1148,2_кузьминки_люблино_часть_ювао,"[кузьминки, люблино, часть, ювао, природ, парк...",[jpg photo ювао природ истор парк кузьминки лю...
3,3,910,3_зелао_зеленоградский_заказник_привязкой,"[зелао, зеленоградский, заказник, привязкой, o...",[jpg photo зелао комплексный заказник зеленогр...
4,4,898,4_дорог_материалами_околоводной_ветвей,"[дорог, материалами, околоводной, ветвей, приб...",[оопт регионального значения природно историче...
5,5,648,5_юзао_img_лес_участок,"[юзао, img, лес, участок, верхняя, химкинский,...",[img jpg photo юзао юао пип битцевский лес уча...
6,6,524,6_доминанты_леса_campanula_chroicocephalus,"[доминанты, леса, campanula, chroicocephalus, ...",[оопт регионального значения ландшафтный заказ...
7,7,516,7_sgcam_жулебинский_portrait_лес,"[sgcam, жулебинский, portrait, лес, участок, ю...",[sgcam jpg photo юзао юао пип битцевский лес у...
8,8,507,8_парковой_антропогенные_объекты_растительностью,"[парковой, антропогенные, объекты, растительно...",[антропогенные объекты окруженные парковой дек...
9,9,465,9_исторический_регионального_озелененная_парк,"[исторический, регионального, озелененная, пар...",[оопт регионального значения природно историче...


In [58]:
model.visualize_topics()

In [59]:
model.visualize_hierarchy()

In [60]:
model.visualize_barchart()

In [61]:
model.get_document_info(final_df['cleaned_text'])

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Representative_document
0,часть территории национального парка лосиный о...,1,1_договор_аренды_правообладателю_земельном,"[договор, аренды, правообладателю, земельном, ...",[публичное акционерное общество московская объ...,договор - аренды - правообладателю - земельном...,False
1,часть территории национального парка лосиный о...,23,23_наследия_культурного_законодательством_феде...,"[наследия, культурного, законодательством, фед...",[оопт регионального значения природно историче...,наследия - культурного - законодательством - ф...,False
2,часть территории национального парка лосиный о...,4,4_дорог_материалами_околоводной_ветвей,"[дорог, материалами, околоводной, ветвей, приб...",[оопт регионального значения природно историче...,дорог - материалами - околоводной - ветвей - п...,False
3,часть территории национального парка лосиный о...,4,4_дорог_материалами_околоводной_ветвей,"[дорог, материалами, околоводной, ветвей, приб...",[оопт регионального значения природно историче...,дорог - материалами - околоводной - ветвей - п...,False
4,часть территории национального парка лосиный о...,4,4_дорог_материалами_околоводной_ветвей,"[дорог, материалами, околоводной, ветвей, приб...",[оопт регионального значения природно историче...,дорог - материалами - околоводной - ветвей - п...,False
...,...,...,...,...,...,...,...
18419,jpg photo зелао комплексный заказник зеленогра...,3,3_зелао_зеленоградский_заказник_привязкой,"[зелао, зеленоградский, заказник, привязкой, o...",[jpg photo зелао комплексный заказник зеленогр...,зелао - зеленоградский - заказник - привязкой ...,True
18420,jpg photo юзао юао пип битцевский лес участок ...,11,11_битцевский_пип_лес_участок,"[битцевский, пип, лес, участок, юао, юзао, oop...",[jpg photo юзао юао пип битцевский лес участок...,битцевский - пип - лес - участок - юао - юзао ...,True
18421,jpg photo юзао юао пип битцевский лес участок ...,11,11_битцевский_пип_лес_участок,"[битцевский, пип, лес, участок, юао, юзао, oop...",[jpg photo юзао юао пип битцевский лес участок...,битцевский - пип - лес - участок - юао - юзао ...,True
18422,jpg photo зелао комплексный заказник зеленогра...,3,3_зелао_зеленоградский_заказник_привязкой,"[зелао, зеленоградский, заказник, привязкой, o...",[jpg photo зелао комплексный заказник зеленогр...,зелао - зеленоградский - заказник - привязкой ...,True
